# RH Analytics — reprodução da limpeza
Rotina principal organizada a partir do notebook do autor. Referência: **25/09/2026**.
Execute a próxima célula e envie `dataset_rh_dashboard-original.xlsx`.
A regra de gênero é exclusiva do conjunto fictício. A saída é CSV UTF-8 separado por `;`.

In [ ]:

# ============================================================
# DATA CLEANING - DATASET DE RH
# ============================================================
#
# Correções realizadas:
# 1. Idade baseada na Data_Nascimento
# 2. Tempo_Empresa_Anos baseado nas datas de admissão/desligamento
# 3. Tempo_Empresa_Anos com 2 casas decimais
# 4. Gênero corrigido com base no primeiro nome
# 5. Verificação de datas inconsistentes
# 6. Verificação final das correções
# 7. Mantém todas as abas originais do Excel
#
# ============================================================


# ============================================================
# 1. IMPORTAR BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files


# ============================================================
# 2. FAZER UPLOAD DO DATASET
# ============================================================

uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

print(f"Arquivo carregado: {arquivo}")


# ============================================================
# 3. VISUALIZAR AS ABAS DO EXCEL
# ============================================================

excel = pd.ExcelFile(arquivo)

print("\nAbas encontradas:")
print(excel.sheet_names)


# ============================================================
# 4. LER A ABA FUNCIONARIOS
# ============================================================

df = pd.read_excel(
    arquivo,
    sheet_name="Funcionarios"
)

print("\nQuantidade de funcionários:", len(df))

print("\nColunas do dataset:")
print(df.columns.tolist())


# ============================================================
# 5. PADRONIZAR COLUNAS DE DATA
# ============================================================

colunas_data = [
    "Data_Nascimento",
    "Data_Admissao",
    "Data_Desligamento",
    "Ultima_Promocao"
]

for coluna in colunas_data:

    if coluna in df.columns:

        df[coluna] = pd.to_datetime(
            df[coluna],
            errors="coerce"
        )


# ============================================================
# 6. DATA DE REFERÊNCIA
# ============================================================
#
# A reprodução utiliza a referência fixa de 25/09/2026.
#
# Funcionário ativo:
# Data atual - Data de admissão
#
# Funcionário desligado:
# Data de desligamento - Data de admissão
#
# ============================================================

data_referencia = pd.Timestamp("2026-09-25")

print("\n========================================")
print("DATA DE REFERÊNCIA")
print("========================================")

print(
    data_referencia.strftime("%d/%m/%Y")
)


# ============================================================
# 7. CORRIGIR IDADE
# ============================================================

print("\n========================================")
print("CORREÇÃO DA IDADE")
print("========================================")


if (
    "Data_Nascimento" in df.columns
    and "Idade" in df.columns
):

    # Guardar idade original
    df["Idade_Original"] = df["Idade"]

    # Calcular idade corretamente
    idade_calculada = (
        data_referencia.year
        - df["Data_Nascimento"].dt.year
        - (
            (df["Data_Nascimento"].dt.month > data_referencia.month) |
            ((df["Data_Nascimento"].dt.month == data_referencia.month) & (df["Data_Nascimento"].dt.day > data_referencia.day))
        ).astype(int)
    )

    # Aplicar a idade calculada
    df["Idade"] = idade_calculada.astype("Int64")

    # Contar alterações
    alteracoes_idade = (
        df["Idade_Original"] != df["Idade"]
    ).sum()

    print(
        f"Idades corrigidas: {alteracoes_idade}"
    )


# ============================================================
# 8. CORRIGIR TEMPO DE EMPRESA
# ============================================================

print("\n========================================")
print("CORREÇÃO DO TEMPO DE EMPRESA")
print("========================================")


if (
    "Data_Admissao" in df.columns
    and "Data_Desligamento" in df.columns
):

    # Guardar valor original
    if "Tempo_Empresa_Anos" in df.columns:

        df["Tempo_Empresa_Anos_Original"] = (
            df["Tempo_Empresa_Anos"]
        )

    # --------------------------------------------------------
    # Para funcionários ativos:
    # usa a data atual
    #
    # Para funcionários desligados:
    # usa a data de desligamento
    # --------------------------------------------------------

    data_final = df["Data_Desligamento"].fillna(
        data_referencia
    )

    # Calcular diferença em dias
    diferenca_dias = (
        data_final - df["Data_Admissao"]
    ).dt.days

    # Converter dias para anos
    #
    # 365.25 considera anos bissextos
    #
    df["Tempo_Empresa_Anos"] = (
        diferenca_dias / 365.25
    )

    # Arredondar para duas casas decimais
    df["Tempo_Empresa_Anos"] = (
        df["Tempo_Empresa_Anos"]
        .round(2)
    )

    print(
        "Tempo de empresa recalculado com sucesso."
    )

    print(
        "Coluna mantida como numérica com 2 casas decimais."
    )


# ============================================================
# 9. CORRIGIR GÊNERO
# ============================================================
#
# A correção utiliza uma tabela explícita para os primeiros
# nomes encontrados no dataset.
#
# Isso evita depender do valor atualmente preenchido na coluna
# Genero, que contém inconsistências.
#
# ============================================================

print("\n========================================")
print("CORREÇÃO DO GÊNERO")
print("========================================")


# Dicionário baseado nos primeiros nomes existentes
# no dataset

genero_por_nome = {

    "Ana": "Feminino",
    "André": "Masculino",
    "Beatriz": "Feminino",
    "Bruno": "Masculino",
    "Caio": "Masculino",
    "Carlos": "Masculino",
    "Daniela": "Feminino",
    "Débora": "Feminino",
    "Eduardo": "Masculino",
    "Felipe": "Masculino",
    "Fernanda": "Feminino",
    "Gabriel": "Masculino",
    "Gabriela": "Feminino",
    "Helena": "Feminino",
    "Henrique": "Masculino",
    "Igor": "Masculino",
    "Isabela": "Feminino",
    "João": "Masculino",
    "Juliana": "Feminino",
    "Larissa": "Feminino",
    "Lucas": "Masculino",
    "Marcelo": "Masculino",
    "Mariana": "Feminino",
    "Natália": "Feminino",
    "Nicolas": "Masculino",
    "Patricia": "Feminino",
    "Rafael": "Masculino",
    "Sabrina": "Feminino",
    "Thiago": "Masculino",
    "Vanessa": "Feminino",
    "William": "Masculino",
    "Yasmin": "Feminino"
}


# Guardar gênero original
df["Genero_Original"] = df["Genero"]


# Extrair primeiro nome
df["Primeiro_Nome"] = (
    df["Nome"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
)


# Aplicar gênero correto
genero_corrigido = (
    df["Primeiro_Nome"]
    .map(genero_por_nome)
)


# Substituir somente onde existe correspondência
df["Genero"] = genero_corrigido.fillna(
    df["Genero"]
)


# Contar quantos registros foram corrigidos
alteracoes_genero = (
    df["Genero_Original"] != df["Genero"]
).sum()


print(
    f"Gêneros corrigidos: {alteracoes_genero}"
)


# Mostrar exemplos que foram corrigidos
correcoes_genero = df[
    df["Genero_Original"] != df["Genero"]
][
    [
        "Nome",
        "Genero_Original",
        "Genero"
    ]
]

print("\nExemplos de correções:")

display(
    correcoes_genero.head(10)
)


# ============================================================
# 10. VERIFICAR DATAS DE NASCIMENTO FUTURAS
# ============================================================

print("\n========================================")
print("VERIFICAÇÃO DAS DATAS")
print("========================================")


nascimentos_futuros = (
    df["Data_Nascimento"]
    > data_referencia
).sum()

print(
    f"Datas de nascimento futuras: "
    f"{nascimentos_futuros}"
)


# ============================================================
# 11. VERIFICAR ADMISSÕES FUTURAS
# ============================================================

admissoes_futuras = (
    df["Data_Admissao"]
    > data_referencia
).sum()

print(
    f"Datas de admissão futuras: "
    f"{admissoes_futuras}"
)


# ============================================================
# 12. VERIFICAR DESLIGAMENTOS FUTUROS
# ============================================================

desligamentos_futuros = (
    df["Data_Desligamento"]
    > data_referencia
).sum()

print(
    f"Datas de desligamento futuras: "
    f"{desligamentos_futuros}"
)


# ============================================================
# 13. VERIFICAR DESLIGAMENTO ANTES DA ADMISSÃO
# ============================================================

desligamento_invalido = (
    df["Data_Desligamento"].notna()
    &
    (
        df["Data_Desligamento"]
        <
        df["Data_Admissao"]
    )
).sum()


print(
    f"Desligamentos anteriores à admissão: "
    f"{desligamento_invalido}"
)


# ============================================================
# 14. VERIFICAÇÃO FINAL DA IDADE
# ============================================================

idade_calculada = (
    data_referencia.year
    - df["Data_Nascimento"].dt.year
    - (
        (df["Data_Nascimento"].dt.month > data_referencia.month) |
        ((df["Data_Nascimento"].dt.month == data_referencia.month) & (df["Data_Nascimento"].dt.day > data_referencia.day))
    ).astype(int)
).astype("Int64")


erros_idade = (
    df["Idade"] != idade_calculada
).sum()


print(
    f"\nInconsistências de idade após correção: "
    f"{erros_idade}"
)


# ============================================================
# 15. VERIFICAÇÃO FINAL DO TEMPO DE EMPRESA
# ============================================================

data_final = df["Data_Desligamento"].fillna(
    data_referencia
)

tempo_calculado = (
    (
        data_final
        - df["Data_Admissao"]
    ).dt.days
    / 365.25
).round(2)


erros_tempo = (
    df["Tempo_Empresa_Anos"]
    != tempo_calculado
).sum()


print(
    f"Inconsistências de tempo de empresa após correção: "
    f"{erros_tempo}"
)


# ============================================================
# 16. VERIFICAÇÃO FINAL DO GÊNERO
# ============================================================

genero_esperado = (
    df["Primeiro_Nome"]
    .map(genero_por_nome)
)


erros_genero = (
    genero_esperado.notna()
    &
    (df["Genero"] != genero_esperado)
).sum()


print(
    f"Inconsistências de gênero após correção: "
    f"{erros_genero}"
)


# ============================================================
# 17. MOSTRAR OS DADOS CORRIGIDOS
# ============================================================

print("\n========================================")
print("DADOS APÓS A LIMPEZA")
print("========================================")


colunas_visualizacao = [
    "ID_Funcionario",
    "Nome",
    "Genero",
    "Data_Nascimento",
    "Idade",
    "Data_Admissao",
    "Data_Desligamento",
    "Tempo_Empresa_Anos"
]


colunas_visualizacao = [
    coluna
    for coluna in colunas_visualizacao
    if coluna in df.columns
]


display(
    df[colunas_visualizacao].head(15)
)


# ============================================================
# 18. INFORMAÇÕES FINAIS
# ============================================================

print("\n========================================")
print("RESUMO FINAL")
print("========================================")


print(
    f"Total de funcionários: {len(df)}"
)


print(
    f"Funcionários ativos: "
    f"{df['Data_Desligamento'].isna().sum()}"
)


print(
    f"Funcionários desligados: "
    f"{df['Data_Desligamento'].notna().sum()}"
)


print(
    f"Idades corrigidas: "
    f"{alteracoes_idade}"
)


print(
    f"Tempos de empresa recalculados: "
    f"{len(df)}"
)


print(
    f"Gêneros corrigidos: "
    f"{alteracoes_genero}"
)


# ============================================================
# 19. REMOVER COLUNAS AUXILIARES
# ============================================================

colunas_auxiliares = [
    "Idade_Original",
    "Tempo_Empresa_Anos_Original",
    "Genero_Original",
    "Primeiro_Nome"
]


for coluna in colunas_auxiliares:

    if coluna in df.columns:

        df.drop(
            columns=coluna,
            inplace=True
        )


# ============================================================
# 20. GARANTIR QUE TEMPO_EMPRESA_ANOS É NUMÉRICO
# ============================================================

df["Tempo_Empresa_Anos"] = pd.to_numeric(
    df["Tempo_Empresa_Anos"],
    errors="coerce"
).round(2)


# ============================================================
# 21. GARANTIR QUE IDADE É INTEIRO
# ============================================================

df["Idade"] = pd.to_numeric(
    df["Idade"],
    errors="coerce"
).astype("Int64")



# Exportar apenas Funcionarios, na ordem do Excel original.
arquivo_saida = "dataset_rh_dashboard-limpo.csv"
df.to_csv(arquivo_saida, sep=";", index=False, encoding="utf-8", date_format="%Y-%m-%d 00:00:00")
files.download(arquivo_saida)
